# Template Genérico de EDA — Exploratory Data Analysis

> **Objetivo:** fornecer um roteiro reutilizável para explorar, compreender e avaliar a qualidade de praticamente qualquer dataset tabular.

Este notebook foi estruturado para responder, de forma progressiva:

1. O que tenho?
2. Como os dados estão estruturados?
3. Os dados são confiáveis?
4. Existem dados ausentes ou duplicados?
5. Como as variáveis se distribuem?
6. Existem outliers?
7. Como as variáveis se relacionam?
8. Existem padrões temporais?
9. Existem diferenças entre grupos?
10. Existe uma variável-alvo?
11. Existem problemas de desbalanceamento ou data leakage?
12. Quais hipóteses e próximos passos surgem da análise?

### Como utilizar

1. Altere apenas a seção **Configuração**.
2. Execute as células sequencialmente.
3. Em análises específicas de domínio, acrescente perguntas e visualizações.
4. Não remova automaticamente outliers ou valores ausentes sem entender sua causa.
5. Se o dataset for utilizado para Machine Learning, faça a análise considerando o contexto da previsão e evitando *data leakage*.

> **Princípio:** EDA não é apenas gerar gráficos. É transformar dados em perguntas, evidências e hipóteses.

## 1. Importação das bibliotecas

In [ ]:
# Bibliotecas principais
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Visualização estatística
import seaborn as sns

# Exibição
from IPython.display import display

# Configurações visuais
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

## 2. Configuração

Esta é a principal seção que você deve adaptar para cada novo dataset.

Preencha:

- caminho do arquivo;
- separador, quando necessário;
- coluna de data, se existir;
- variável-alvo, se existir;
- colunas que devem ser ignoradas na análise;
- colunas categóricas e numéricas, caso seja necessário sobrescrever a detecção automática.

In [ ]:
# =========================
# CONFIGURAÇÃO DO DATASET
# =========================

CAMINHO_ARQUIVO = "dataset.csv"

# Formato: "csv", "excel", "parquet"
FORMATO_ARQUIVO = "csv"

# Para CSV:
SEPARADOR = ","
ENCODING = "utf-8"

# Informe a coluna temporal, se existir.
# Exemplo: "data_venda"
COLUNA_DATA = None

# Informe a variável-alvo se estiver trabalhando com Machine Learning.
# Exemplo: "inadimplente"
TARGET = None

# Colunas que não devem participar de determinadas análises.
COLUNAS_IGNORAR = []

# Deixe como None para detecção automática.
COLUNAS_NUMERICAS = None
COLUNAS_CATEGORICAS = None

## 3. Carregamento dos dados

In [ ]:
# =========================
# CARREGAMENTO
# =========================

if FORMATO_ARQUIVO.lower() == "csv":
    df = pd.read_csv(
        CAMINHO_ARQUIVO,
        sep=SEPARADOR,
        encoding=ENCODING
    )

elif FORMATO_ARQUIVO.lower() in ["xlsx", "xls", "excel"]:
    df = pd.read_excel(CAMINHO_ARQUIVO)

elif FORMATO_ARQUIVO.lower() == "parquet":
    df = pd.read_parquet(CAMINHO_ARQUIVO)

else:
    raise ValueError("FORMATO_ARQUIVO deve ser csv, excel ou parquet.")

print(f"Dataset carregado com sucesso.")
print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]:,}")

## 4. Visão geral do dataset

### Perguntas

- Quantas linhas e colunas existem?
- O que representa uma linha?
- Quais são as primeiras e últimas observações?
- Quais são os tipos de dados?
- Existem colunas inesperadas?

In [ ]:
print("Dimensões:", df.shape)

display(df.head())
display(df.tail())

print("\nInformações:")
df.info()

## 5. Inventário das variáveis

In [ ]:
inventario = pd.DataFrame({
    "coluna": df.columns,
    "tipo": df.dtypes.astype(str),
    "n_unicos": df.nunique(dropna=True).values,
    "nulos": df.isna().sum().values,
    "%_nulos": (df.isna().mean() * 100).round(2).values
})

inventario["%_unicos"] = (
    inventario["n_unicos"] / len(df) * 100
).round(2)

display(inventario.sort_values("%_nulos", ascending=False))

## 6. Estatísticas descritivas

Para variáveis numéricas:

- mínimo;
- máximo;
- média;
- mediana;
- quartis;
- desvio padrão.

Para categóricas:

- quantidade;
- frequência;
- categorias mais comuns.

> **A média sozinha raramente é suficiente.** Compare média e mediana e observe a distribuição.

In [ ]:
display(df.describe(include="all").T)

## 7. Dados ausentes

### Perguntas

- Quais colunas possuem valores ausentes?
- Qual o percentual de ausência?
- A ausência está concentrada em alguma variável ou grupo?
- `0`, string vazia e `NaN` possuem o mesmo significado?

In [ ]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "%_nulos": (df.isna().mean() * 100).round(2)
})

display(nulos.sort_values("nulos", ascending=False))

In [ ]:
# Visualização simples dos valores ausentes

nulos_plot = nulos[nulos["nulos"] > 0].sort_values("nulos", ascending=True)

if len(nulos_plot) > 0:
    plt.figure(figsize=(10, 6))
    plt.barh(nulos_plot.index, nulos_plot["%_nulos"])
    plt.xlabel("% de valores ausentes")
    plt.title("Valores ausentes por variável")
    plt.show()
else:
    print("Não foram encontrados valores ausentes.")

## 8. Duplicidades

### Perguntas

- Existem linhas completamente duplicadas?
- Existem identificadores duplicados?
- Duplicatas representam erro ou eventos legítimos?

In [ ]:
duplicadas = df.duplicated().sum()

print(f"Linhas duplicadas: {duplicadas:,}")
print(f"Percentual: {duplicadas / len(df) * 100:.2f}%")

if duplicadas > 0:
    display(df[df.duplicated(keep=False)].head(20))

## 9. Tipos de dados e consistência

In [ ]:
# Resumo dos tipos
display(
    pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "exemplos": [
            df[col].dropna().iloc[0] if df[col].notna().any() else None
            for col in df.columns
        ]
    })
)

### Conversão opcional de datas

Se houver uma coluna de data, converta-a explicitamente.

```python
df["data"] = pd.to_datetime(df["data"], errors="coerce")
```

`errors="coerce"` transforma valores inválidos em `NaT`, permitindo identificá-los posteriormente.

In [ ]:
if COLUNA_DATA is not None and COLUNA_DATA in df.columns:
    df[COLUNA_DATA] = pd.to_datetime(df[COLUNA_DATA], errors="coerce")
    print(df[COLUNA_DATA].dtype)
else:
    print("Nenhuma coluna de data configurada.")

## 10. Variáveis numéricas

### Perguntas

- Como cada variável é distribuída?
- Existem valores extremos?
- Há assimetria?
- Existem variáveis quase constantes?
- Existem valores impossíveis para o domínio?

In [ ]:
if COLUNAS_NUMERICAS is None:
    colunas_numericas = df.select_dtypes(include=np.number).columns.tolist()
else:
    colunas_numericas = [
        c for c in COLUNAS_NUMERICAS if c in df.columns
    ]

colunas_numericas = [
    c for c in colunas_numericas
    if c not in COLUNAS_IGNORAR
]

print(f"{len(colunas_numericas)} variáveis numéricas:")
print(colunas_numericas)

In [ ]:
# Histogramas das variáveis numéricas

if colunas_numericas:
    df[colunas_numericas].hist(
        figsize=(16, 12),
        bins=30
    )
    plt.suptitle("Distribuição das variáveis numéricas", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Boxplots individuais

for col in colunas_numericas:
    plt.figure(figsize=(10, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot — {col}")
    plt.show()

## 11. Identificação de outliers pelo IQR

O método do **Intervalo Interquartil (IQR)** identifica observações abaixo de `Q1 - 1.5 × IQR` ou acima de `Q3 + 1.5 × IQR`.

> Isso é um método de detecção, não uma autorização para excluir registros.

In [ ]:
outliers_resumo = []

for col in colunas_numericas:
    serie = df[col].dropna()

    if serie.empty:
        continue

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    mask = (df[col] < limite_inferior) | (df[col] > limite_superior)

    outliers_resumo.append({
        "variavel": col,
        "Q1": q1,
        "Q3": q3,
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "n_outliers": mask.sum(),
        "%_outliers": mask.mean() * 100
    })

outliers_df = pd.DataFrame(outliers_resumo)

if not outliers_df.empty:
    display(outliers_df.sort_values("n_outliers", ascending=False))

## 12. Variáveis categóricas

### Perguntas

- Quais categorias existem?
- Quais são as mais frequentes?
- Existem categorias raras?
- Existem erros de grafia ou categorias equivalentes?
- Há forte desbalanceamento?

In [ ]:
if COLUNAS_CATEGORICAS is None:
    colunas_categoricas = df.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()
else:
    colunas_categoricas = [
        c for c in COLUNAS_CATEGORICAS if c in df.columns
    ]

colunas_categoricas = [
    c for c in colunas_categoricas
    if c not in COLUNAS_IGNORAR
]

print(f"{len(colunas_categoricas)} variáveis categóricas:")
print(colunas_categoricas)

In [ ]:
for col in colunas_categoricas:
    print("=" * 70)
    print(f"Variável: {col}")
    print(f"Categorias únicas: {df[col].nunique(dropna=True):,}")
    display(
        pd.DataFrame({
            "contagem": df[col].value_counts(dropna=False),
            "percentual": (
                df[col].value_counts(normalize=True, dropna=False) * 100
            ).round(2)
        }).head(20)
    )

## 13. Correlação entre variáveis numéricas

### Perguntas

- Existem variáveis fortemente correlacionadas?
- Há possíveis variáveis redundantes?
- Existem relações positivas ou negativas?

> Correlação indica associação, não causalidade.

In [ ]:
if len(colunas_numericas) >= 2:
    corr = df[colunas_numericas].corr()

    plt.figure(figsize=(12, 8))
    sns.heatmap(
        corr,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0
    )
    plt.title("Matriz de correlação")
    plt.show()
else:
    print("São necessárias pelo menos duas variáveis numéricas.")

## 14. Relações entre variáveis numéricas

In [ ]:
# Scatterplots de pares de variáveis.
# Para datasets com muitas colunas, selecione manualmente algumas.

PARES_NUMERICOS = []

# Exemplo:
# PARES_NUMERICOS = [
#     ("idade", "salario"),
#     ("preco", "quantidade")
# ]

for x, y in PARES_NUMERICOS:
    if x in df.columns and y in df.columns:
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=df, x=x, y=y)
        plt.title(f"{x} × {y}")
        plt.show()

## 15. Análise por grupos

Perguntas:

- Como uma variável numérica muda entre categorias?
- Existem diferenças entre grupos?
- O tamanho dos grupos é comparável?
- A média está sendo influenciada por outliers?

In [ ]:
# Configure manualmente para cada dataset.

COLUNA_GRUPO = None
COLUNA_METRICA = None

if (
    COLUNA_GRUPO in df.columns
    if COLUNA_GRUPO is not None
    else False
) and (
    COLUNA_METRICA in df.columns
    if COLUNA_METRICA is not None
    else False
):
    resumo_grupos = (
        df.groupby(COLUNA_GRUPO)[COLUNA_METRICA]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .sort_values("count", ascending=False)
    )

    display(resumo_grupos)
else:
    print("Configure COLUNA_GRUPO e COLUNA_METRICA para executar esta análise.")

## 16. Análise temporal

Utilize esta seção somente quando houver uma variável de data/hora.

### Perguntas

- Existe tendência?
- Existe sazonalidade?
- Existem picos e quedas?
- Existem períodos sem dados?
- Há mudanças estruturais ao longo do tempo?

In [ ]:
if COLUNA_DATA is not None and COLUNA_DATA in df.columns:

    df["_ano"] = df[COLUNA_DATA].dt.year
    df["_mes"] = df[COLUNA_DATA].dt.month
    df["_dia_semana"] = df[COLUNA_DATA].dt.day_name()
    df["_data"] = df[COLUNA_DATA].dt.date

    print("Período:")
    print("Início:", df[COLUNA_DATA].min())
    print("Fim:", df[COLUNA_DATA].max())

    print("\nQuantidade de registros por ano:")
    display(df["_ano"].value_counts().sort_index())

    plt.figure(figsize=(14, 5))
    df.groupby("_data").size().plot()
    plt.title("Quantidade de registros ao longo do tempo")
    plt.xlabel("Data")
    plt.ylabel("Quantidade")
    plt.show()

else:
    print("Nenhuma coluna temporal foi configurada.")

## 17. Distribuição da variável-alvo

Se o dataset for utilizado em Machine Learning, configure `TARGET`.

Perguntas:

- O target é numérico ou categórico?
- Em classificação, as classes estão balanceadas?
- Existem classes raras?
- Como as features se comportam em relação ao target?

In [ ]:
if TARGET is not None and TARGET in df.columns:

    print("Target:", TARGET)
    print("Tipo:", df[TARGET].dtype)
    print("Valores únicos:", df[TARGET].nunique())

    if (
        df[TARGET].dtype == "object"
        or str(df[TARGET].dtype) == "category"
        or df[TARGET].nunique() <= 20
    ):
        display(
            pd.DataFrame({
                "contagem": df[TARGET].value_counts(dropna=False),
                "percentual": (
                    df[TARGET]
                    .value_counts(normalize=True, dropna=False) * 100
                ).round(2)
            })
        )

        plt.figure(figsize=(10, 5))
        sns.countplot(data=df, x=TARGET)
        plt.title(f"Distribuição do target — {TARGET}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        display(df[TARGET].describe())

        plt.figure(figsize=(10, 5))
        sns.histplot(df[TARGET].dropna(), kde=True)
        plt.title(f"Distribuição do target — {TARGET}")
        plt.show()

else:
    print("Nenhum TARGET foi configurado.")

## 18. Relação das variáveis com o target

Esta seção gera análises genéricas. Para um projeto real, selecione as relações mais relevantes para o problema.

In [ ]:
if TARGET is not None and TARGET in df.columns:

    # Numéricas x target categórico
    if df[TARGET].nunique() <= 20:

        for col in colunas_numericas[:10]:
            if col == TARGET:
                continue

            plt.figure(figsize=(10, 5))
            sns.boxplot(data=df, x=TARGET, y=col)
            plt.title(f"{col} por classe de {TARGET}")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

        # Categóricas x target
        for col in colunas_categoricas[:10]:
            if col == TARGET:
                continue

            tabela = pd.crosstab(
                df[col],
                df[TARGET],
                normalize="index"
            ) * 100

            display(tabela.head(20))

    else:
        print("Target possui muitas categorias ou é contínuo.")
else:
    print("Configure TARGET para executar esta seção.")

## 19. Verificação conceitual de Data Leakage

Antes de treinar qualquer modelo, pergunte:

> **Esta variável estaria disponível no momento em que a previsão fosse realizada?**

Sinais de alerta:

- coluna que contém informação posterior ao evento;
- resultado final usado para prever o próprio resultado;
- identificadores que codificam diretamente o target;
- informações calculadas utilizando todo o dataset;
- variáveis criadas depois do evento que queremos prever.

### Checklist

- [ ] Conheço o momento em que a previsão será feita?
- [ ] Conheço o momento em que cada feature é produzida?
- [ ] Alguma feature utiliza informação do futuro?
- [ ] Alguma agregação foi calculada usando dados que deveriam estar fora da amostra?
- [ ] O pré-processamento será ajustado somente no conjunto de treino?

## 20. Variáveis com cardinalidade alta

Identifique variáveis categóricas com muitas categorias. Elas podem exigir tratamento especial em análises ou modelos.

In [ ]:
cardinalidade = pd.DataFrame({
    "variavel": df.columns,
    "n_unicos": [df[c].nunique(dropna=True) for c in df.columns],
    "%_unicos": [
        df[c].nunique(dropna=True) / len(df) * 100
        for c in df.columns
    ]
}).sort_values("n_unicos", ascending=False)

display(cardinalidade.head(30))

## 21. Variáveis quase constantes

Uma variável que possui praticamente um único valor pode ter pouca utilidade analítica ou preditiva.

> Isso não significa automaticamente que ela deve ser removida: seu significado de domínio deve ser considerado.

In [ ]:
quase_constantes = []

for col in df.columns:
    frequencias = df[col].value_counts(normalize=True, dropna=False)

    if len(frequencias) > 0:
        proporcao_maior = frequencias.iloc[0]

        quase_constantes.append({
            "variavel": col,
            "valor_mais_frequente": frequencias.index[0],
            "proporcao": proporcao_maior * 100,
            "n_unicos": df[col].nunique(dropna=True)
        })

quase_constantes_df = pd.DataFrame(quase_constantes)

display(
    quase_constantes_df
    .sort_values("proporcao", ascending=False)
    .head(30)
)

## 22. Perguntas específicas do domínio

Esta seção é propositalmente manual. A EDA deve incorporar conhecimento do problema.

### Exemplos

**Vendas**
- Quais produtos geram maior receita?
- Existe sazonalidade?
- Quais regiões apresentam maior volume?

**Educação**
- Frequência está associada ao desempenho?
- Existem diferenças entre turmas?
- Quais fatores aparecem associados à evasão?

**Criminalidade**
- Onde as ocorrências se concentram?
- Quando ocorrem?
- Existem padrões temporais ou geográficos?

**Financeiro**
- Existem transações anômalas?
- Quais categorias concentram maior valor?
- Existem mudanças ao longo do tempo?

## 23. Registro de hipóteses

Use esta célula para documentar hipóteses que surgiram durante a EDA.

Exemplo:

- **Hipótese:** vendas aumentam nos finais de semana.
- **Evidência:** média de vendas sábado/domingo é maior.
- **Próxima análise:** comparar diferentes meses e controlar por sazonalidade.

In [ ]:
# ==========================================
# HIPÓTESES E PERGUNTAS PARA INVESTIGAÇÃO
# ==========================================

hipoteses = [
    # {
    #     "hipotese": "Exemplo de hipótese",
    #     "evidencia": "Exemplo de evidência encontrada",
    #     "proxima_analise": "Como verificar melhor"
    # }
]

hipoteses_df = pd.DataFrame(hipoteses)

if not hipoteses_df.empty:
    display(hipoteses_df)
else:
    print("Nenhuma hipótese registrada ainda.")

## 24. Checklist final da EDA

Antes de encerrar, verifique cada item.

In [ ]:
checklist = {
    "Estrutura compreendida": False,
    "Tipos de dados verificados": False,
    "Valores ausentes analisados": False,
    "Duplicidades analisadas": False,
    "Consistência dos valores verificada": False,
    "Variáveis numéricas exploradas": False,
    "Variáveis categóricas exploradas": False,
    "Outliers investigados": False,
    "Correlações analisadas": False,
    "Relações entre grupos analisadas": False,
    "Análise temporal realizada (quando aplicável)": False,
    "Target analisado (quando aplicável)": False,
    "Data leakage investigado (quando aplicável)": False,
    "Perguntas específicas do domínio investigadas": False,
    "Hipóteses registradas": False
}

checklist_df = pd.DataFrame(
    list(checklist.items()),
    columns=["item", "concluido"]
)

display(checklist_df)

# 25. Conclusões da EDA

Ao finalizar, procure responder:

### Sobre os dados
- O que cada observação representa?
- Quais são as principais variáveis?
- Os dados possuem problemas de qualidade?

### Sobre as distribuições
- Quais variáveis apresentam maior variabilidade?
- Existem assimetrias?
- Existem outliers relevantes?

### Sobre relações
- Quais relações parecem mais importantes?
- Existem diferenças entre grupos?
- Existem padrões temporais ou geográficos?

### Sobre Machine Learning
- Qual é o target?
- Existe desbalanceamento?
- Existem features potencialmente relevantes?
- Existe risco de data leakage?
- É necessário criar novas features?

### Sobre o próximo passo
A EDA deve terminar com uma decisão sobre o que investigar ou fazer a seguir:

**Limpeza → Feature Engineering → Visualização aprofundada → Testes estatísticos → Machine Learning → Modelagem preditiva → Comunicação dos resultados**

> **Regra prática:** uma boa EDA não precisa responder tudo. Ela precisa revelar o que vale a pena investigar a seguir.

# Referência rápida — Fluxo universal

```text
                 DATASET
                    │
                    ▼
          ┌──────────────────┐
          │ 1. ESTRUTURA     │
          │ shape / info     │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ 2. QUALIDADE     │
          │ NaN / duplicados │
          │ tipos / valores  │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ 3. UNIVARIADA    │
          │ distribuições    │
          │ categorias       │
          │ outliers         │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ 4. BIVARIADA     │
          │ relações         │
          │ grupos           │
          │ correlações      │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ 5. MULTIVARIADA  │
          │ padrões          │
          │ interações       │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ 6. TEMPO/ESPAÇO  │
          │ quando aplicável │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ 7. TARGET        │
          │ quando aplicável │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ 8. HIPÓTESES     │
          │ perguntas novas  │
          └────────┬─────────┘
                   ▼
             PRÓXIMO PASSO
```